In [7]:
import pandas as pd
import numpy as np
data_path = 'D:/Datasets/UserBehavior.csv' 
# 读取前 200 万行数据
columns = ['user_id', 'item_id', 'category_id', 'behavior_type', 'timestamp']
df = pd.read_csv(data_path, names=columns, nrows=2_000_000)
# 打印数据形状和预览
print("数据形状:", df.shape)
df.head()

数据形状: (2000000, 5)


,user_id,item_id,category_id,behavior_type,timestamp
0,1,2268318,2520377,pv,1511544070
1,1,2333346,2520771,pv,1511561733
2,1,2576651,149192,pv,1511572885
3,1,3830808,4181361,pv,1511593493
4,1,4365585,2520377,pv,1511596146


In [8]:
# 1. 时间戳转换（转化为北京时间）
df['datetime'] = pd.to_datetime(df['timestamp'], unit='s') + pd.Timedelta(hours=8)

# 2. 提取日期和小时
df['date'] = df['datetime'].dt.date
df['hour'] = df['datetime'].dt.hour

# 3. 检查时间范围（必须落在 2017-11-25 到 2017-12-03 之间）
print("数据起始日期:", df['date'].min())
print("数据结束日期:", df['date'].max())

# 4. 检查重复行并去重
duplicate_count = df.duplicated().sum()
print(f"发现重复行数: {duplicate_count}")
df = df.drop_duplicates()

# 5. 行为类型分布统计（打个底，看下各类行为占比）
behavior_counts = df['behavior_type'].value_counts()
print("各行为分布情况：\n", behavior_counts)

# 6. 把行为映射为数字（为后面漏斗排序做准备）
behavior_map = {'pv': 1, 'fav': 2, 'cart': 3, 'buy': 4}
df['behavior_code'] = df['behavior_type'].map(behavior_map)

# 查看处理后的数据
df.head()

数据起始日期: 1970-01-01
数据结束日期: 2018-08-28
发现重复行数: 0
各行为分布情况：
 behavior_type
pv      1791216
cart     111015
fav       57526
buy       40243
Name: count, dtype: int64


,user_id,item_id,category_id,behavior_type,timestamp,datetime,date,hour,behavior_code
0,1,2268318,2520377,pv,1511544070,2017-11-25 01:21:10,2017-11-25,1,1
1,1,2333346,2520771,pv,1511561733,2017-11-25 06:15:33,2017-11-25,6,1
2,1,2576651,149192,pv,1511572885,2017-11-25 09:21:25,2017-11-25,9,1
3,1,3830808,4181361,pv,1511593493,2017-11-25 15:04:53,2017-11-25,15,1
4,1,4365585,2520377,pv,1511596146,2017-11-25 15:49:06,2017-11-25,15,1


In [ ]:
# 按 user_id 聚合
user_df = df.groupby('user_id').agg(
    pv_count=('behavior_type', lambda x: (x == 'pv').sum()),
    fav_count=('behavior_type', lambda x: (x == 'fav').sum()),
    cart_count=('behavior_type', lambda x: (x == 'cart').sum()),
    buy_count=('behavior_type', lambda x: (x == 'buy').sum()),
    last_buy_date=('date', lambda x: x[df.loc[x.index, 'behavior_type'] == 'buy'].max() if 'buy' in x.values else pd.NaT)
).reset_index()

# 计算 Recency（假设数据集的最后一天 2017-12-03 是分析截止日）
analysis_date = pd.to_datetime('2017-12-03').date()
user_df['R'] = user_df['last_buy_date'].apply(
    lambda x: (analysis_date - x).days if pd.notna(x) else 99
)

# 计算 F 和 M
user_df['F'] = user_df['buy_count']
user_df['M'] = user_df['buy_count'] # 数据集没有金额，用购买次数代理 M

# 过滤出至少有浏览记录的用户，排除极端异常（比如纯僵尸号）
user_df = user_df[user_df['pv_count'] > 0]

print("用户宽表维度:", user_df.shape)
user_df.head()